# ETL by hand: from raw data to a dashboard table

**Scenario.** ShopNow (the shop from the BI lecture) wants a daily funnel dashboard:

> sessions → add to cart → orders, per day and per traffic source (`utm_source`), plus bounce rate, conversion rate and revenue in EUR.

Two source systems deliver raw data:

- `raw_web_events`: the tracking tool writes one row per click (`page_view`, `add_to_cart`)
- `raw_orders`: the shop backend writes one row per order, in the customer's currency

Your job: build the table the dashboard needs. **You are the ETL process.**

**Time: 12 minutes, in pairs.** Get as far as you can, stop when time is up. The point is to *feel* what happens between a raw source and a dashboard. We discuss the solution afterwards.

Pandas you need (from day 1): `pd.to_datetime`, `.groupby().agg()`, `.merge()`, boolean columns.

**Stuck on the how? Ask an AI now.** Every step has a hint with the pandas function you need. If the hint is not enough, paste the step and the `.head()` output into ChatGPT, Claude or Gemini. Then check the answer against the raw table: does the row count make sense, does the date look like a date?

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

# ---------------------------------------------------------------
# Raw data as it arrives from the two source systems.
# Nothing to change here, just run the cell.
# ---------------------------------------------------------------
sources = ["paid", "organic", "email"]
events_rows, orders_rows = [], []
order_no = 1000

for day, day_str in [(1, "2024/10/01"), (2, "2024/10/02")]:
    for i in range(45):
        session_id = f"s-{day}{i:02d}"
        user_id = f"u-{rng.integers(1, 40)}"
        utm = sources[rng.integers(0, 3)]
        hour, minute = rng.integers(8, 22), rng.integers(0, 60)
        n_views = rng.integers(1, 5)
        t = hour * 60 + minute
        for v in range(n_views):
            events_rows.append([f"{day_str} {t // 60:02d}:{t % 60:02d}", session_id, user_id, "page_view", utm])
            t += rng.integers(1, 4)
        # add to cart happens in ~55 % of the sessions that have more than one page view
        if n_views > 1 and rng.random() < 0.55:
            events_rows.append([f"{day_str} {t // 60:02d}:{t % 60:02d}", session_id, user_id, "add_to_cart", utm])
            t += rng.integers(1, 4)
            # 50-70 % of those sessions place an order (paid traffic converts a bit worse)
            if rng.random() < (0.50 if utm == "paid" else 0.70):
                order_no += 1
                currency = "USD" if rng.random() < 0.35 else "EUR"
                cents = int(rng.integers(1500, 25000))
                orders_rows.append([f"2024-10-0{day}T{t // 60:02d}:{t % 60:02d}Z", f"O-{order_no}", user_id, session_id, cents, currency])

raw_web_events = pd.DataFrame(events_rows, columns=["event_time", "session_id", "user_id", "event", "utm_source"])
raw_orders = pd.DataFrame(orders_rows, columns=["order_time", "order_id", "user_id", "session_id", "total_cents", "currency"])

# exchange rate the finance team wants us to use
EUR_PER_USD = 0.91

print(f"{len(raw_web_events)} web events, {len(raw_orders)} orders")

## Task 1: Extract (2 min)

Look at both raw tables. Find **three things that are messy** and would break a dashboard if used directly.

Hints: look at the time formats, the number of rows per session, the money columns.

Start with `.dtypes` (next cell): if a column says `object` or `str`, it is text, not a date, even when it looks like one.

In [ ]:
raw_web_events.head(10)

In [ ]:
raw_orders.head()

In [ ]:
# everything object (or str)? then event_time and order_time are text, not dates
print(raw_web_events.dtypes, raw_orders.dtypes, sep="\n\n")

**Write your three findings here:**

1. ...
2. ...
3. ...

## Task 2: Transform to staging tables (6 min)

Staging tables are **clean and reusable**: one row per business object (session, order), proper types, one currency.

### 2a) `stg_sessions`: one row per session

| session_id | date | utm_source | events | add_to_cart | bounce |
|---|---|---|---|---|---|
| s-100 | 2024-10-01 | paid | 3 | 1 | 0 |

- `date`: the day of the session (from `event_time`)
- `events`: number of rows (clicks) in the session
- `add_to_cart`: 1 if the session contains an `add_to_cart` event, else 0
- `bounce`: 1 if the session has exactly one event, else 0

In [ ]:
events = raw_web_events.copy()

# Step 1: parse the time column into a real datetime and extract the date
# hint: pd.to_datetime(events["event_time"], format="%Y/%m/%d %H:%M").dt.date
events["date"] = ...

# Step 2: a helper column that is True for add_to_cart rows
# hint: events["event"] == "add_to_cart"
events["is_add_to_cart"] = ...

# Step 3: one row per session: groupby session_id and aggregate
# hint: .groupby("session_id").agg(date=("date", "min"), utm_source=("utm_source", "first"), events=("event", "count"), add_to_cart=("is_add_to_cart", "max"))
stg_sessions = ...

# Step 4: bounce = exactly one event
# hint: (stg_sessions["events"] == 1).astype(int)
stg_sessions["bounce"] = ...

stg_sessions = stg_sessions.reset_index()
stg_sessions.head()

### 2b) `stg_orders`: one row per order, revenue in EUR

| order_id | date | session_id | revenue_eur |
|---|---|---|---|
| O-1001 | 2024-10-01 | s-100 | 118.21 |

Use `EUR_PER_USD` for USD orders. EUR orders stay as they are.

In [ ]:
orders = raw_orders.copy()

# Step 1: parse order_time (ISO format) and extract the date
# hint: pd.to_datetime(orders["order_time"]).dt.date
orders["date"] = ...

# Step 2: revenue in EUR: cents -> currency units -> EUR
# hint: np.where(orders["currency"] == "USD", ... * EUR_PER_USD, ...)
orders["revenue_eur"] = ...

stg_orders = orders[["order_id", "date", "session_id", "revenue_eur"]]
stg_orders.head()

## Task 3: Load the mart table (4 min)

The dashboard needs exactly this table, nothing more:

| date | utm_source | sessions | bounces | add_to_cart | orders | revenue_eur | conversion |
|---|---|---|---|---|---|---|---|
| 2024-10-01 | paid | 15 | 4 | 6 | 2 | 231.40 | 13.3 % |

`conversion` = orders / sessions.

In [ ]:
# Step 1: orders and revenue per session (one row per session, 0 if no order)
# hint: stg_orders.groupby("session_id").agg(orders=("order_id", "count"), revenue_eur=("revenue_eur", "sum"))
orders_per_session = ...

# Step 2: merge into the sessions table (left join, fill missing with 0)
# hint: stg_sessions.merge(orders_per_session, on="session_id", how="left").fillna({"orders": 0, "revenue_eur": 0})
sessions_with_orders = ...

# Step 3: aggregate per date and utm_source
# hint: .groupby(["date", "utm_source"]).agg(sessions=("session_id", "count"), bounces=("bounce", "sum"), add_to_cart=("add_to_cart", "sum"), orders=("orders", "sum"), revenue_eur=("revenue_eur", "sum"))
mart_daily_funnel = ...

# Step 4: conversion rate in percent
# hint: 100 * mart_daily_funnel["orders"] / mart_daily_funnel["sessions"], then .round(1)
mart_daily_funnel["conversion"] = ...

mart_daily_funnel = mart_daily_funnel.reset_index()
mart_daily_funnel

## Bonus: the dashboard chart

One line, now that the mart exists. This is why marts are built: the chart code is trivial.

In [ ]:
mart_daily_funnel.pivot(index="date", columns="utm_source", values="conversion").plot.bar(ylabel="conversion in %", title="Conversion per traffic source");

## Task 4: Reflect (write 1 sentence each, we discuss in the plenum)

1. The shop starts selling in Switzerland (CHF). Which of your steps breaks, and where would you notice it?
2. Finance wants a monthly revenue report per currency. Which table do they start from: raw, staging or mart? Why?
3. Where would you put an automatic test so that a broken tracking tool is noticed *before* the dashboard shows wrong numbers?

## What you just did has names

| Your step | Name | Where it lives |
|---|---|---|
| Looked at the two source tables | **Extract** | source systems (tracking, shop backend) |
| Parsed dates, converted currency, one row per session | **Transform** | staging tables |
| Built the one table the dashboard needs | **Load** | data mart in the data warehouse |

Back to the slides.